# Notebook 11 — Ecology and Population Biology

**Module:** 05 — Biology Fundamentals  
**Notebook:** 11 of 18  
**Prerequisites:** NB08  
**Time estimate:** 60 minutes

---
## Step 1 — Motivation

Ecological models — Lotka-Volterra predator-prey, logistic growth, food web
network analysis — form the mathematical backbone of computational ecology and
systems biology. If you plan to work in environmental genomics, metagenomics,
or multi-species systems biology, these are foundational.

---
## Step 2 — Intuition

Ecology studies how populations interact with each other and with their environment.
At its core: populations grow when resources allow, decline when predators or disease
intervene, and reach equilibria (or oscillate) depending on the balance of forces.
The same mathematics — differential equations, matrix models, graph theory — applies
to ecological networks and gene regulatory networks.

---
## Step 3 — Biological Background

**Key concepts:**

- **Carrying capacity (K):** maximum population size the environment can sustain
- **Intrinsic growth rate (r):** per-capita rate of population increase when resources are unlimited
- **Trophic level:** position in a food chain (producers → primary consumers → secondary → top predators)
- **Keystone species:** a species with disproportionate impact relative to its abundance

**Logistic growth:**
$$\frac{dN}{dt} = rN\left(1 - \frac{N}{K}\right)$$

- When N << K: exponential growth (≈ rN)
- When N = K: no growth (equilibrium)
- When N > K: decline

**Lotka-Volterra predator-prey:**
$$\frac{dV}{dt} = \alpha V - \beta VP$$
$$\frac{dP}{dt} = \delta VP - \gamma P$$

V = prey (victim) population, P = predator. These coupled equations produce
oscillations — cycles of predator and prey abundance.

**Food web as a graph:**
- Nodes: species
- Directed edges: 'is eaten by' (or 'eats')
- Adjacency matrix A[i,j] = 1 if species i eats species j
- Trophic level = longest path from a producer (plant/bacterium) to the node

---
## Step 6 — Python Implementation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

In [ ]:
# Cell 6.1 — Logistic growth model
def logistic_growth(t, N, r, K):
    return r * N * (1 - N / K)

r, K = 0.3, 1000.0
sol = solve_ivp(logistic_growth, [0, 50], [10.0], args=(r, K),
                dense_output=True, max_step=0.5)
t_log = np.linspace(0, 50, 500)
N_log = sol.sol(t_log)[0]

print(f"Logistic growth: r={r}, K={K}, N0=10")
print(f"  N at t=20: {N_log[np.argmin(abs(t_log-20))]:.1f}")
print(f"  N at t=50: {N_log[-1]:.1f}  (K={K})")

In [ ]:
# Cell 6.2 — Lotka-Volterra predator-prey
def lotka_volterra(t, y, alpha, beta, delta, gamma):
    V, P = y
    dV = alpha * V - beta * V * P
    dP = delta * V * P - gamma * P
    return [dV, dP]

# Classic parameters (rabbits and foxes)
alpha = 0.8   # rabbit birth rate
beta  = 0.02  # predation rate
delta = 0.01  # fox birth rate per rabbit eaten
gamma = 0.5   # fox death rate

sol_lv = solve_ivp(lotka_volterra, [0, 60], [40, 9], args=(alpha, beta, delta, gamma),
                   dense_output=True, max_step=0.05)
t_lv = np.linspace(0, 60, 1200)
V_t, P_t = sol_lv.sol(t_lv)

print(f"Lotka-Volterra predator-prey oscillations:")
print(f"  Prey equilibrium: P* = gamma/delta = {gamma/delta:.1f}")
print(f"  Pred equilibrium: V* = alpha/beta  = {alpha/beta:.1f}")

In [ ]:
# Cell 6.3 — Food web as adjacency matrix
# Simple rocky intertidal food web
species = ['Algae', 'Mussels', 'Barnacles', 'Whelks', 'Sea stars', 'Limpets']
# adj[i,j] = 1 if species i is eaten by species j
adj = np.array([
    [0, 1, 1, 0, 0, 1],  # Algae eaten by Mussels, Barnacles, Limpets
    [0, 0, 0, 1, 1, 0],  # Mussels eaten by Whelks, Sea stars
    [0, 0, 0, 1, 1, 0],  # Barnacles eaten by Whelks, Sea stars
    [0, 0, 0, 0, 0, 0],  # Whelks (top predator)
    [0, 0, 0, 0, 0, 0],  # Sea stars (top predator)
    [0, 0, 0, 0, 0, 0],  # Limpets (herbivore)
], dtype=float)

in_degree  = adj.sum(axis=0)   # how many species eat each species (= 'prey' degree)
out_degree = adj.sum(axis=1)   # how many things each species eats (= 'predator' degree)

print("Food web adjacency analysis:")
for i, sp in enumerate(species):
    print(f"  {sp:<12}  preys on {out_degree[i]:.0f} others;  eaten by {in_degree[i]:.0f} others")

---
## Step 7 — Visualization

In [ ]:
# Cell 7.1 — Logistic growth + Lotka-Volterra oscillations
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Panel 1: Logistic growth
ax = axes[0]
ax.plot(t_log, N_log, color='steelblue', lw=2)
ax.axhline(K, color='gray', ls='--', lw=1, label=f'K = {K:.0f}')
ax.set_xlabel('Time'); ax.set_ylabel('Population size N')
ax.set_title(f'Logistic growth: r={r}, K={K}')
ax.legend()

# Panel 2: Lotka-Volterra time series
ax = axes[1]
ax.plot(t_lv, V_t, color='seagreen', lw=2, label='Prey (rabbits)')
ax.plot(t_lv, P_t, color='tomato', lw=2, label='Predators (foxes)')
ax.set_xlabel('Time'); ax.set_ylabel('Population size')
ax.set_title('Lotka-Volterra predator-prey cycles')
ax.legend()

plt.tight_layout()
plt.show()

---
## Step 8 — Exercises

1. Implement `logistic_growth(t, N, r, K)` and solve it using `scipy.integrate.solve_ivp`.
   Plot it for r=0.1, 0.3, 0.8 (same K=1000) on the same axes.
2. The Lotka-Volterra model produces perfect cycles. Real predator-prey systems often
   damp to a stable equilibrium. Name one biological mechanism that could cause damping.
3. Remove the sea star from the food web adjacency matrix (set its row and column to 0).
   Which species would you predict to increase? This is the Pisaster experiment — look
   it up and see what actually happened.
4. The allee effect describes populations that struggle to grow at very low density
   (e.g., difficulty finding mates). Modify the logistic growth equation to include
   this effect. What happens near N=0?

---
## Quiz — Active Recall

1. What does the carrying capacity K represent in logistic growth?
2. Why do predator-prey systems oscillate? What causes the cycle?
3. What is a trophic level? How many trophic levels does most food energy travel through
   before being lost?
4. What is a keystone species? How is this concept relevant to conservation genomics?
5. How is the food web adjacency matrix similar to the gene regulatory network adjacency
   matrix? Name one property they share.

---
## Reflection

**Date completed:** ____________________

> *[The gut microbiome is a community of ~1000 bacterial species. Which ecological
> concepts from this notebook would apply to studying how antibiotic treatment
> disrupts and reshapes the microbiome community?]*

---
*Next: `12_human_genetics_and_disease.ipynb`*